# Recommender Experiment
## Setup and load processed data

In [1]:
## Recommender experiment ##
"""This notebook reproduces the final analyses reported in the paper:
1. chronological warm-up/train/validation/test split;
2. fixed warm-up historical features;
3. final 400,000-episode Logistic Regression models;
4. 20,000-episode validation and final test evaluation;
5. customer-clustered bootstrap confidence intervals;
6. hard-negative robustness check;
7. post-hoc episode-type, size, concentration, Spearman/FDR and controlled-regression analyses;
8. LightGBM robustness check on the fresh 38,085-episode holdout;
9. Logistic Regression and LightGBM feature importance;
10. LensKit metric verification.

Run 01_data_preparation_eda.ipynb first.
"""
# The preliminary 200k development run and debugging/inspection cells from the working notebook are intentionally omitted 
# because they do not feed the final reported models or tables.

from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from scipy.stats import spearmanr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import lightgbm as lgb
from lightgbm import LGBMRanker

RANDOM_SEED = 42

project_root = Path.cwd().resolve()

if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent
if not (project_root / "data" / "processed").exists():
    raise FileNotFoundError(
        "Could not find data/processed. Run 01_data_preparation_eda.ipynb first."
    )

DATA_PROCESSED = project_root / "data" / "processed"
RESULTS_DIR = project_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("LightGBM:", lgb.__version__)

LightGBM: 4.7.0


In [2]:
# Loading the processed datasets
transactions = pd.read_parquet(DATA_PROCESSED / "transactions_online.parquet")

episodes = pd.read_parquet(DATA_PROCESSED / "multi-item_shopping_episodes.parquet")

article_features = pd.read_parquet(DATA_PROCESSED / "article_features.parquet")

group_stats = pd.read_parquet(DATA_PROCESSED / "group_stats.parquet")

## Chronological split

In [3]:
## EXPERIMENTAL EPISODES AND CHRONOLOGICAL SPLIT ##

# For item recommendation, requiring at least 2 unique items in a shopping
# episode, otherwise the episode is not useful for training a recommender model.
experiment_episodes = episodes[episodes["unique_articles"] >= 2].copy()
experiment_episodes = (
    experiment_episodes
    .sort_values(["t_dat", "customer_id"])
    .reset_index(drop=True)
)

experiment_episodes["episode_id"] = np.arange(
    len(experiment_episodes),
    dtype=np.int64
)

print(f"Experimental episodes: {len(experiment_episodes):,}")  # 3,895,481

Experimental episodes: 3,895,481


In [4]:
# Chronological splitting of the episodes into warm-up, training, 
# validation, and test sets. (20% warm-up, 60% training, 10% validation, 10% test)

episodes_per_day = (
    experiment_episodes
    .groupby("t_dat")
    .size()
    .sort_index()
)

# Cumulative sum of episodes per day
cumulative_share = episodes_per_day.cumsum() / episodes_per_day.sum()

# Finding dates for splitting the episodes into warm-up, training, validation,
#  and test sets 
# 0-20 % Warm-up, 20-80 % Training, 80-90 % Validation, 90-100 % Test

warmup_end = cumulative_share[
    cumulative_share >= 0.2].index[0]

train_end = cumulative_share[
    cumulative_share >= 0.80
].index[0]

validation_end = cumulative_share[
    cumulative_share >= 0.90
].index[0]

# Creating the four episode sets
warmup_episodes = experiment_episodes[
    experiment_episodes["t_dat"] <= warmup_end
].copy()

train_episodes = experiment_episodes[
    (experiment_episodes["t_dat"] > warmup_end)
    & (experiment_episodes["t_dat"] <= train_end)
].copy()

validation_episodes = experiment_episodes[
    (experiment_episodes["t_dat"] > train_end)
    & (experiment_episodes["t_dat"] <= validation_end)
].copy()

test_episodes = experiment_episodes[
    experiment_episodes["t_dat"] > validation_end
].copy()

In [5]:
# Checking the number of episodes in each set
splits = {
    "Warm-up": warmup_episodes,
    "Train": train_episodes,
    "Validation": validation_episodes,
    "Test": test_episodes
}

for name, df in splits.items():
    print(
        f"{name:10s}: "
        f"{len(df):>10,} episodes | "
        f"{len(df) / len(experiment_episodes):>6.2%} | "
        f"{df['t_dat'].min().date()} "
        f"to {df['t_dat'].max().date()}"
    )

Warm-up   :    780,970 episodes | 20.05% | 2018-09-20 to 2019-02-23
Train     :  2,338,922 episodes | 60.04% | 2019-02-24 to 2020-05-05
Validation:    388,783 episodes |  9.98% | 2020-05-06 to 2020-07-04
Test      :    386,806 episodes |  9.93% | 2020-07-05 to 2020-09-22


In [6]:
# Verifying there are no overlapping episodes between the four sets
print(
    warmup_episodes["t_dat"].max()
    < train_episodes["t_dat"].min()
)

print(
    train_episodes["t_dat"].max()
    < validation_episodes["t_dat"].min()
)

print(
    validation_episodes["t_dat"].max()
    < test_episodes["t_dat"].min()
)
# All three should return True

True
True
True


In [7]:
# Saving the four splits to the processed data directory
warmup_episodes.to_parquet(
    DATA_PROCESSED / "warmup_episodes.parquet",
    index=False
)

train_episodes.to_parquet(
    DATA_PROCESSED / "train_episodes.parquet",
    index=False
)

validation_episodes.to_parquet(
    DATA_PROCESSED / "validation_episodes.parquet",
    index=False
)

test_episodes.to_parquet(
    DATA_PROCESSED / "test_episodes.parquet",
    index=False
)

In [8]:
# Split dates
split_dates = pd.DataFrame({
    "boundary": [
        "warmup_end",
        "train_end",
        "validation_end"
    ],
    "date": [
        warmup_end,
        train_end,
        validation_end
    ]
})

split_dates

,boundary,date
0,warmup_end,2019-02-23
1,train_end,2020-05-05
2,validation_end,2020-07-04


In [9]:
## Constructing consumer preferences using the initial historical feature period ##
# (warm-up period) using the original online transactions dataset, including
# all online purchases, including single-item shopping episodes.
# A customer´s previous single-item purchases still tell us something about their
# preferences, even if they are not useful for training a recommender model.

warmup_transactions = transactions[
    transactions["t_dat"] <= warmup_end     
].copy()

warmup_transactions.to_parquet(
    DATA_PROCESSED / "warmup_transactions.parquet",
    index=False
)

## Feature construction

In [10]:
## Historical feature construction ##
# Creating historical features from the warm-up transactions dataset

# Since the article features dataset does not contain the article´s price,
# we will use the warm-up transactions dataset to calculate the average price
# of each article, which will be used as a historical feature for the recommender model
# and representing the article´s price as negative candidates.
# comparing with the average price paid by the customer.
# (price_distance feature) 

history = warmup_transactions.merge(
    article_features,
    on="article_id",
    how="left",
    validate="many_to_one"
)

# Customer-level features
customer_stats = (
    history.groupby("customer_id")
    .agg(customer_average_price=("price", "mean"),
         customer_total_purchases=("article_id", "size")   
).reset_index()
)
# Item popularity and historical candidate price
# Item popularity = number of times the article was historically purchased by 
# any customer.
# Candidate average price = average price for the article in the warm-up period.

# For every article:
item_stats = (
    history
    .groupby("article_id")
    .agg(
        item_popularity=("article_id", "size"),
        candidate_average_price=("price", "mean")
    )
    .reset_index()
)

# Customer-item count (0 for never purchased, 1 for purchased once, 2 for purchased twice, etc.)
customer_item_stats = (
    history
    .groupby(["customer_id", "article_id"])
    .size()
    .rename("customer_item_count")
    .reset_index()
)

In [11]:
# Preference variables

# Preference (u, c) = customer purchases belonging to candidate category
#                       / all historical purchases by customer

# example:
# customer_product_group_pref = historical purchases in candidate product group
#                               / all historical purchases by customer

# Specifying the preference variables to be created:
PREFERENCE_FEATURES = {
    "product_code": "customer_product_code_pref",
    "product_type_no": "customer_product_type_pref",
    "product_group_name": "customer_product_group_pref",
    "graphical_appearance_name": "customer_appearance_pref",
    "colour_group_code": "customer_colour_pref",
    "department_no": "customer_department_pref",
    "section_no": "customer_section_pref",
    "garment_group_no": "customer_garment_group_pref"
}

# Generic function:
def create_customer_preference(history_df, attribute, preference_name):
    counts = (
        history_df.groupby(["customer_id", attribute])
        .size()
        .rename("customer_attribute_count")
        .reset_index()
    )
    totals = (
        history_df.groupby("customer_id")
        .size()
        .rename("customer_total_purchases")
        .reset_index()
    )
    out = counts.merge(totals, on="customer_id", how="left")
    out[preference_name] = (
        out["customer_attribute_count"] / out["customer_total_purchases"]
    )
    return out[["customer_id", attribute, preference_name]]

preference_tables = {
    attribute: create_customer_preference(history, attribute, feature_name)
    for attribute, feature_name in PREFERENCE_FEATURES.items()
}


In [ ]:
candidate_article_features = article_features.rename(
    columns={"article_id": "candidate_article"}
)

candidate_item_stats = item_stats.rename(
    columns={"article_id": "candidate_article"}
)

candidate_customer_item_stats = customer_item_stats.rename(
    columns={"article_id": "candidate_article"}
)

candidate_catalog = item_stats["article_id"].to_numpy()
candidate_catalog_set = set(candidate_catalog)
warmup_customers = set(customer_stats["customer_id"])

print(f"Warm-up candidate catalogue: {len(candidate_catalog):,} articles")  # 45,168
print(f"Customers with warm-up history: {len(warmup_customers):,}")         # 484,500

Warm-up candidate catalogue: 45,168 articles
Customers with warm-up history: 484,500


## Feature sets and reusable functions

In [13]:
## Feature sets and reusable functions ##
INDEPENDENT_FEATURES = [
    "customer_average_price",
    "item_popularity",
    "customer_item_count",
    "customer_product_code_pref",
    "customer_product_type_pref",
    "customer_product_group_pref",
    "customer_appearance_pref",
    "customer_colour_pref",
    "customer_department_pref",
    "customer_section_pref",
    "customer_garment_group_pref",
    "price_distance",
]

CONTEXT_ATTRIBUTES = {
    "product_code": "same_product_code_share",
    "product_type_no": "same_product_type_share",
    "product_group_name": "same_product_group_share",
    "graphical_appearance_name": "same_appearance_share",
    "colour_group_code": "same_colour_share",
    "department_no": "same_department_share",
    "section_no": "same_section_share",
    "garment_group_no": "same_garment_share",
}

CONTEXT_FEATURES = list(CONTEXT_ATTRIBUTES.values())
EPISODE_CONTEXT_FEATURES = INDEPENDENT_FEATURES + CONTEXT_FEATURES

print("Independent-choice features:", len(INDEPENDENT_FEATURES))   # 12
print("Episode-context features:", len(EPISODE_CONTEXT_FEATURES))  # 20

Independent-choice features: 12
Episode-context features: 20


In [14]:
# Hold out one article from each training episode,
# removing all occurrences of the held-out article ID from the context, 
# (otherwise the bracket-aware model literally sees the target article in its context)

def prepare_recommender_episodes(
        episodes_df,
        candidate_catalog_set,
        seed=42
):
    rng = np.random.default_rng(seed)
    prepared_rows = []

    for row in episodes_df.itertuples(index = False):
        # Preserve duplicate purchases
        articles = list(row.articles)

        # Unique article IDs, preserving their original order
        unique_articles = list(dict.fromkeys(articles))

        # Target must have historical information from warm-up
        eligible_targets = [
            article
            for article in unique_articles
            if article in candidate_catalog_set
        ]

        # No historically known article available as target
        if len(eligible_targets) == 0:
            continue

        # Randomly choose one eligible article as target
        target = rng.choice(eligible_targets)

        # Remove all occurrences of the target from context
        context = [
            article
            for article in articles
            if article != target
        ]

        if len(context) == 0: # should not happen since we have episode sizes of >= 2 articles, but safeguard
            continue

        prepared_rows.append({
            "episode_id": row.episode_id,
            "customer_id": row.customer_id,
            "t_dat": row.t_dat,
            "target_article": target,
            "context_articles": context,
            "episode_size": len(articles),
            "context_size": len(context)
            
        })

    return pd.DataFrame(prepared_rows)

# Negative candidate sampler:
def sample_negatives(
        target,
        context,
        candidate_catalog,
        n_negatives,
        rng
):

    """ Sample unique negative candidate articles that are not the target and
    not already present in the visible context. """

    excluded = set(context)
    excluded.add(target)

    negatives = []
    selected = set()

    while len(negatives) < n_negatives:
        n_needed = n_negatives - len(negatives)

        # Draw a few extra to reduce number of loops
        sampled = rng.choice(
            candidate_catalog,
            size = max(n_needed * 2, 20),
            replace =  True
        )

        for article in sampled:
            if (
                article not in excluded
                and article not in selected
            ):
                negatives.append(article)
                selected.add(article)

            if len(negatives) == n_negatives:
                break

    return negatives

# Building the candidate table:
def build_candidate_df(
    prepared_episodes,
    candidate_catalog,
    n_negatives=10,
    seed=42
):
    rng = np.random.default_rng(seed)

    episode_ids = []
    customer_ids = []
    candidate_articles = []
    labels = []

    for row in prepared_episodes.itertuples(index=False):

        negatives = sample_negatives(
            target=row.target_article,
            context=row.context_articles,
            candidate_catalog=candidate_catalog,
            n_negatives=n_negatives,
            rng=rng
        )

        # Positive candidate
        episode_ids.append(row.episode_id)
        customer_ids.append(row.customer_id)
        candidate_articles.append(row.target_article)
        labels.append(1)

        # Negative candidates
        for article in negatives:
            episode_ids.append(row.episode_id)
            customer_ids.append(row.customer_id)
            candidate_articles.append(article)
            labels.append(0)

    return pd.DataFrame({
        "episode_id": episode_ids,
        "customer_id": customer_ids,
        "candidate_article": candidate_articles,
        "label": labels
    })


In [15]:
# Feature-merging procedure
# Defining reusable function:

def add_independent_features(candidate_df):
    df = candidate_df.copy()

    # Candidate metadata
    df = df.merge(
        candidate_article_features,
        on = "candidate_article",
        how = "left",
        validate = "many_to_one"
    )

    # Customer historical statistics
    df = df.merge(
        customer_stats,
        on = "customer_id",
        how = "left",
        validate = "many_to_one"
    )

    # Candidate historical statistics
    df = df.merge(
        candidate_item_stats,
        on = "candidate_article",
        how = "left",
        validate = "many_to_one"
    )

    # Customer-item count
    df = df.merge(
        candidate_customer_item_stats,
        on = [
            "customer_id",
            "candidate_article"
        ],
        how = "left",
        validate = "many_to_one"
    )

    df["customer_item_count"] = (
        df["customer_item_count"]
        .fillna(0)
        .astype("int32")
    )

    # Customer attribute preferences
    for attribute, feature_name in (
        PREFERENCE_FEATURES.items()
    ):
        df = df.merge(
            preference_tables[attribute],
            on = [
                "customer_id",
                attribute
            ],
            how = "left",
            validate = "many_to_one"
        )

        df[feature_name] = (
            df[feature_name]
            .fillna(0)
        )

    df["price_distance"] = (
        df["candidate_average_price"]
        - df["customer_average_price"]
    ).abs()

    return df

# Reusable function for making context features
def add_context_features(
    feature_df,
    prepared_episodes
):
    df = feature_df.copy()

    context_items = (
        prepared_episodes[
            ["episode_id", "context_articles"]
        ]
        .explode("context_articles")
        .rename(
            columns={
                "context_articles": "article_id"
            }
        )
    )

    context_items = context_items.merge(
        article_features,
        on="article_id",
        how="left",
        validate="many_to_one"
    )

    context_sizes = prepared_episodes[
        ["episode_id", "context_size"]
    ]

    df = df.merge(
        context_sizes,
        on="episode_id",
        how="left",
        validate="many_to_one"
    )

    for attribute, feature_name in (
        CONTEXT_ATTRIBUTES.items()
    ):

        counts = (
            context_items
            .groupby(
                ["episode_id", attribute],
                dropna=False
            )
            .size()
            .rename("context_match_count")
            .reset_index()
        )

        df = df.merge(
            counts,
            on=["episode_id", attribute],
            how="left",
            validate="many_to_one"
        )

        df["context_match_count"] = (
            df["context_match_count"]
            .fillna(0)
        )

        df[feature_name] = (
            df["context_match_count"]
            / df["context_size"]
        )

        df = df.drop(
            columns="context_match_count"
        )

    return df


In [16]:
def calculate_ranking_metrics(ranks, k=10):
    ranks = np.asarray(ranks)
    hit = (ranks <= k).astype(float)
    mrr = np.where(ranks <= k, 1 / ranks, 0)
    ndcg = np.where(ranks <= k, 1 / np.log2(ranks + 1), 0)
    return hit, mrr, ndcg


def score_and_evaluate(
    feature_df,
    independent_model,
    context_model,
    independent_features=INDEPENDENT_FEATURES,
    context_features=EPISODE_CONTEXT_FEATURES,
):
    scores = feature_df[
        ["episode_id", "customer_id", "candidate_article", "label"]
    ].copy()

    scores["independent_score"] = independent_model.predict_proba(
        feature_df[independent_features]
    )[:, 1]

    scores["context_score"] = context_model.predict_proba(
        feature_df[context_features]
    )[:, 1]

    for model in ["independent", "context"]:
        scores[f"{model}_rank"] = (
            scores.groupby("episode_id")[f"{model}_score"]
            .rank(ascending=False, method="first")
        )

    targets = scores.loc[scores["label"] == 1].copy()

    for model in ["independent", "context"]:
        hit, mrr, ndcg = calculate_ranking_metrics(
            targets[f"{model}_rank"], k=10
        )
        targets[f"{model}_hit10"] = hit
        targets[f"{model}_mrr10"] = mrr
        targets[f"{model}_ndcg10"] = ndcg

    results = pd.DataFrame({
        "Independent": [
            targets["independent_hit10"].mean(),
            targets["independent_mrr10"].mean(),
            targets["independent_ndcg10"].mean(),
        ],
        "Episode-context": [
            targets["context_hit10"].mean(),
            targets["context_mrr10"].mean(),
            targets["context_ndcg10"].mean(),
        ],
    }, index=["Hit@10", "MRR@10", "NDCG@10"])

    results["Absolute difference"] = (
        results["Episode-context"] - results["Independent"]
    )

    return scores, targets, results


def add_metric_differences(targets):
    targets = targets.copy()
    targets["hit10_difference"] = (
        targets["context_hit10"] - targets["independent_hit10"]
    )
    targets["mrr10_difference"] = (
        targets["context_mrr10"] - targets["independent_mrr10"]
    )
    targets["ndcg10_difference"] = (
        targets["context_ndcg10"] - targets["independent_ndcg10"]
    )
    return targets


def customer_cluster_bootstrap(
    df,
    difference_col,
    customer_col="customer_id",
    n_bootstrap=1000,
    seed=42,
):
    customer_level = (
        df.groupby(customer_col)[difference_col]
        .agg(["sum", "count"])
    )

    sums = customer_level["sum"].to_numpy()
    counts = customer_level["count"].to_numpy()
    n_customers = len(customer_level)

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        idx = rng.integers(0, n_customers, size=n_customers)
        bootstrap_means[i] = sums[idx].sum() / counts[idx].sum()

    observed = df[difference_col].mean()
    lower, upper = np.percentile(bootstrap_means, [2.5, 97.5])
    return observed, lower, upper


def clustered_ci_table(targets):
    targets = add_metric_differences(targets)
    rows = []

    for metric, column in {
        "Hit@10": "hit10_difference",
        "MRR@10": "mrr10_difference",
        "NDCG@10": "ndcg10_difference",
    }.items():
        difference, lower, upper = customer_cluster_bootstrap(
            targets,
            difference_col=column,
            n_bootstrap=1000,
            seed=RANDOM_SEED,
        )
        rows.append({
            "Metric": metric,
            "Difference": difference,
            "CI lower": lower,
            "CI upper": upper,
        })

    return targets, pd.DataFrame(rows)

## Preparing training, validation and test episodes

In [17]:
## Prepare training, validation and test samples ##

train_episodes_warm = train_episodes[
    train_episodes["customer_id"].isin(warmup_customers)
].copy()

validation_episodes_warm = validation_episodes[
    validation_episodes["customer_id"].isin(warmup_customers)
].copy()

test_episodes_warm = test_episodes[
    test_episodes["customer_id"].isin(warmup_customers)
].copy()

train_prepared = prepare_recommender_episodes(
    train_episodes_warm, candidate_catalog_set, seed=RANDOM_SEED
)
validation_prepared = prepare_recommender_episodes(
    validation_episodes_warm, candidate_catalog_set, seed=RANDOM_SEED
)
test_prepared = prepare_recommender_episodes(
    test_episodes_warm, candidate_catalog_set, seed=RANDOM_SEED
)

print("After warm-up-customer filtering:")
print(
    f"Train {len(train_episodes_warm):,} | "  # 1,590,349
    f"Validation {len(validation_episodes_warm):,} | "   # 222,804
    f"Test {len(test_episodes_warm):,}"  # 232,067
)
print("After eligible-target preparation:")
print(
    f"Train {len(train_prepared):,} | " # 973,400
    f"Validation {len(validation_prepared):,} | "  # 62,127
    f"Test {len(test_prepared):,}"   # 58,085
)

After warm-up-customer filtering:
Train 1,590,349 | Validation 222,804 | Test 232,067
After eligible-target preparation:
Train 973,400 | Validation 62,127 | Test 58,085


In [18]:
# Checking that the target never occurs in its context:
target_in_context = train_prepared.apply(
    lambda row:
        row["target_article"] in row["context_articles"],
    axis=1
)

target_in_context.value_counts()  # False for all 973,400 episodes

False    973400
Name: count, dtype: int64

In [19]:
N_TRAIN_EPISODES = 400_000
N_VALIDATION_EPISODES = 20_000
N_TEST_EPISODES = 20_000

train_prepared_400k = train_prepared.sample(
    n=N_TRAIN_EPISODES,
    random_state=RANDOM_SEED,
).copy()

validation_prepared_sample = validation_prepared.sample(
    n=N_VALIDATION_EPISODES,
    random_state=RANDOM_SEED,
).copy()

test_prepared_sample = test_prepared.sample(
    n=N_TEST_EPISODES,
    random_state=RANDOM_SEED,
).copy()

print(
    len(train_prepared_400k),             # 400,000
    len(validation_prepared_sample),      # 20,000
    len(test_prepared_sample),            # 20,000
)

400000 20000 20000


## Candidate generation

In [20]:
## Candidate generation and feature construction ##
train_candidates_400k = build_candidate_df(
    train_prepared_400k,
    candidate_catalog,
    n_negatives=10,
    seed=RANDOM_SEED,
)

validation_candidates = build_candidate_df(
    validation_prepared_sample,
    candidate_catalog,
    n_negatives=99,
    seed=RANDOM_SEED,
)

test_candidates = build_candidate_df(
    test_prepared_sample,
    candidate_catalog,
    n_negatives=99,
    seed=RANDOM_SEED,
)

assert train_candidates_400k.groupby("episode_id").size().eq(11).all()
assert validation_candidates.groupby("episode_id").size().eq(100).all()
assert test_candidates.groupby("episode_id").size().eq(100).all()

for candidate_df in [
    train_candidates_400k,
    validation_candidates,
    test_candidates,
]:
    assert candidate_df.groupby("episode_id")["label"].sum().eq(1).all()
    assert not candidate_df.duplicated(
        ["episode_id", "candidate_article"]
    ).any()

In [21]:
# Saving the dataset with training candidates:
# IMPORTANT! Both models must see exactly these same examples! Both models use train_candidates_400k

train_candidates_400k.to_parquet(
    DATA_PROCESSED / "train_candidates.parquet",
    index = False
)

In [22]:
train_features_400k = add_context_features(
    add_independent_features(train_candidates_400k),
    train_prepared_400k,
)

validation_features = add_context_features(
    add_independent_features(validation_candidates),
    validation_prepared_sample,
)

test_features = add_context_features(
    add_independent_features(test_candidates),
    test_prepared_sample,
)

for df in [train_features_400k, validation_features, test_features]:
    assert not df[EPISODE_CONTEXT_FEATURES].isna().any().any()

In [23]:
# Saving the training_features dataset with both independent and context features
train_features_400k.to_parquet(
    DATA_PROCESSED / "train_features.parquet",
    index = False
)

## Logistic Regression models - defining and training

In [ ]:
# Building final Logistic Regression models identically (same preprocessing/model specifications)
# Defining models
# Independent baseline model:
independent_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter = 1000,
            random_state = RANDOM_SEED
        )
    )
])

# Episode-context model:
context_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter = 1000,
            random_state = RANDOM_SEED
        )
    )
])

# Training the models
independent_model.fit(
    train_features_400k[INDEPENDENT_FEATURES],
    train_features_400k["label"],
)

context_model.fit(
    train_features_400k[EPISODE_CONTEXT_FEATURES],
    train_features_400k["label"],
)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['customer_average_price','item_popularity','customer_item_count',..., 'same_department_share','same_section_share','same_garment_share']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


## LR Validation

In [25]:
# Saving validation candidates (both models must use these exactly same candidates!)
validation_candidates.to_parquet(
    DATA_PROCESSED / "validation_candidates.parquet",
    index = False
)

In [26]:
# Saving the validation features table
validation_features.to_parquet(
    DATA_PROCESSED / "validation_features.parquet",
    index = False
)

In [27]:
## Validation results ##

validation_scores, validation_targets, validation_results = score_and_evaluate(
    validation_features,
    independent_model,
    context_model,
)

validation_targets, validation_cluster_ci = clustered_ci_table(
    validation_targets
)

validation_results

,Independent,Episode-context,Absolute difference
Hit@10,0.536150,0.730950,0.194800
MRR@10,0.265127,0.488766,0.223640
NDCG@10,0.328555,0.546428,0.217874


In [28]:
validation_cluster_ci

,Metric,Difference,CI lower,CI upper
0,Hit@10,0.194800,0.186871,0.202351
1,MRR@10,0.223640,0.217954,0.229340
2,NDCG@10,0.217874,0.212336,0.223458


## LR Testing

In [29]:
## Final Logistic Regression test results
test_scores, test_targets, test_results = score_and_evaluate(
    test_features,
    independent_model,
    context_model,
)

test_targets, test_cluster_ci = clustered_ci_table(test_targets)

test_results

,Independent,Episode-context,Absolute difference
Hit@10,0.605600,0.765200,0.159600
MRR@10,0.335476,0.527718,0.192241
NDCG@10,0.399060,0.584285,0.185224


In [30]:
test_cluster_ci

,Metric,Difference,CI lower,CI upper
0,Hit@10,0.159600,0.152920,0.165949
1,MRR@10,0.192241,0.186486,0.197506
2,NDCG@10,0.185224,0.179890,0.190088


In [31]:
# Saving the recommender models, test results, targets and scores
joblib.dump(
    independent_model,
    RESULTS_DIR / "independent_model_400k.joblib",
)
joblib.dump(
    context_model,
    RESULTS_DIR / "episode_context_model_400k.joblib",
)

test_results.to_csv(RESULTS_DIR / "lr_final_test_results.csv")
test_cluster_ci.to_csv(
    RESULTS_DIR / "lr_final_test_clustered_ci.csv",
    index=False,
)
test_scores.to_parquet(
    RESULTS_DIR / "lr_final_test_scores.parquet",
    index=False,
)
test_targets.to_parquet(
    RESULTS_DIR / "lr_final_test_targets.parquet",
    index=False,
)

## Hard-negative robustness check

In [32]:
## Hard-negative robustness check ##
catalog_metadata = (
    article_features.loc[
        article_features["article_id"].isin(candidate_catalog_set),
        ["article_id", "product_group_name"],
    ]
    .drop_duplicates("article_id")
    .copy()
)

article_to_product_group = (
    catalog_metadata
    .set_index("article_id")["product_group_name"]
    .to_dict()
)

product_group_pools = {
    group: group_df["article_id"].to_numpy()
    for group, group_df in catalog_metadata.groupby(
        "product_group_name",
        dropna=True,
    )
}


def sample_random_negatives(candidate_catalog, excluded, n_negatives, rng):
    selected = set()

    while len(selected) < n_negatives:
        needed = n_negatives - len(selected)
        sampled = rng.choice(
            candidate_catalog,
            size=max(needed * 2, 20),
            replace=True,
        )

        for article in sampled:
            if article not in excluded and article not in selected:
                selected.add(article)
            if len(selected) == n_negatives:
                break

    return list(selected)


def build_hard_negative_candidates(
    prepared_episodes,
    candidate_catalog,
    article_to_product_group,
    product_group_pools,
    n_hard=50,
    total_negatives=99,
    seed=42,
):
    rng = np.random.default_rng(seed)
    rows = []

    for row in prepared_episodes.itertuples(index=False):
        target = row.target_article
        excluded = set(row.context_articles)
        excluded.add(target)

        target_group = article_to_product_group.get(target)
        hard_negatives = []

        if target_group in product_group_pools:
            eligible_hard = [
                article for article in product_group_pools[target_group]
                if article not in excluded
            ]
            n_to_sample = min(n_hard, len(eligible_hard))
            if n_to_sample:
                hard_negatives = rng.choice(
                    eligible_hard,
                    size=n_to_sample,
                    replace=False,
                ).tolist()

        random_negatives = sample_random_negatives(
            candidate_catalog,
            excluded | set(hard_negatives),
            total_negatives - len(hard_negatives),
            rng,
        )

        rows.append({
            "episode_id": row.episode_id,
            "customer_id": row.customer_id,
            "candidate_article": target,
            "label": 1,
            "negative_type": "positive",
        })

        rows.extend({
            "episode_id": row.episode_id,
            "customer_id": row.customer_id,
            "candidate_article": article,
            "label": 0,
            "negative_type": "same_product_group",
        } for article in hard_negatives)

        rows.extend({
            "episode_id": row.episode_id,
            "customer_id": row.customer_id,
            "candidate_article": article,
            "label": 0,
            "negative_type": "random",
        } for article in random_negatives)

    return pd.DataFrame(rows)

## Generating hard candidates

In [33]:
hard_candidates = build_hard_negative_candidates(
    validation_prepared_sample,
    candidate_catalog,
    article_to_product_group,
    product_group_pools,
    n_hard=50,
    total_negatives=99,
    seed=RANDOM_SEED,
)

assert hard_candidates.groupby("episode_id").size().eq(100).all()
assert hard_candidates.groupby("episode_id")["label"].sum().eq(1).all()

# Check: How many hard negatives did each episode receive?
hard_counts = (
    hard_candidates[
        hard_candidates[
            "negative_type"
        ] == "same_product_group"
    ]
    .groupby("episode_id")
    .size()
)
#hard_counts.describe()

hard_features = add_context_features(
    add_independent_features(hard_candidates),
    validation_prepared_sample,
)

hard_scores, hard_targets, hard_results = score_and_evaluate(
    hard_features,
    independent_model,
    context_model,
)

hard_targets, hard_cluster_ci = clustered_ci_table(hard_targets)

hard_results

,Independent,Episode-context,Absolute difference
Hit@10,0.511950,0.677850,0.165900
MRR@10,0.245392,0.410997,0.165605
NDCG@10,0.307772,0.473942,0.166170


In [34]:
hard_cluster_ci

,Metric,Difference,CI lower,CI upper
0,Hit@10,0.165900,0.158791,0.172542
1,MRR@10,0.165605,0.160604,0.170677
2,NDCG@10,0.166170,0.161214,0.171134


## Comparison

In [35]:
# Comparison:
hard_negative_comparison = pd.DataFrame({
    "Setting": ["Random negatives", "Hard negatives"],
    "Delta Hit@10": [
        validation_cluster_ci.loc[
            validation_cluster_ci["Metric"] == "Hit@10", "Difference"
        ].iloc[0],
        hard_cluster_ci.loc[
            hard_cluster_ci["Metric"] == "Hit@10", "Difference"
        ].iloc[0],
    ],
    "Delta MRR@10": [
        validation_cluster_ci.loc[
            validation_cluster_ci["Metric"] == "MRR@10", "Difference"
        ].iloc[0],
        hard_cluster_ci.loc[
            hard_cluster_ci["Metric"] == "MRR@10", "Difference"
        ].iloc[0],
    ],
    "Delta NDCG@10": [
        validation_cluster_ci.loc[
            validation_cluster_ci["Metric"] == "NDCG@10", "Difference"
        ].iloc[0],
        hard_cluster_ci.loc[
            hard_cluster_ci["Metric"] == "NDCG@10", "Difference"
        ].iloc[0],
    ],
    "NDCG CI lower": [
        validation_cluster_ci.loc[
            validation_cluster_ci["Metric"] == "NDCG@10", "CI lower"
        ].iloc[0],
        hard_cluster_ci.loc[
            hard_cluster_ci["Metric"] == "NDCG@10", "CI lower"
        ].iloc[0],
    ],
    "NDCG CI upper": [
        validation_cluster_ci.loc[
            validation_cluster_ci["Metric"] == "NDCG@10", "CI upper"
        ].iloc[0],
        hard_cluster_ci.loc[
            hard_cluster_ci["Metric"] == "NDCG@10", "CI upper"
        ].iloc[0],
    ],
})

hard_negative_comparison

,Setting,Delta Hit@10,Delta MRR@10,Delta NDCG@10,NDCG CI lower,NDCG CI upper
0,Random negatives,0.1948,0.223640,0.217874,0.212336,0.223458
1,Hard negatives,0.1659,0.165605,0.166170,0.161214,0.171134


## Post-hoc test episode analyses

In [36]:
## Post-hoc episode-structure analyses ## test episodes
CONCENTRATION_COLUMNS = [
    "unique_article_concentration",
    "product_code_concentration",
    "product_type_concentration",
    "product_group_concentration",
    "colour_concentration",
    "department_concentration",
    "graphical_appearance_concentration",
    "section_concentration",
    "garment_group_concentration",
]

episode_characteristics = group_stats[
    ["customer_id", "t_dat", "episode_type"] + CONCENTRATION_COLUMNS
].copy()

episode_characteristics["t_dat"] = pd.to_datetime(
    episode_characteristics["t_dat"]
)
test_prepared_sample["t_dat"] = pd.to_datetime(
    test_prepared_sample["t_dat"]
)

test_episode_analysis = (
    test_prepared_sample
    .merge(
        test_targets[
            [
                "episode_id",
                "independent_ndcg10",
                "context_ndcg10",
                "ndcg10_difference",
            ]
        ],
        on="episode_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        episode_characteristics,
        on=["customer_id", "t_dat"],
        how="left",
        validate="one_to_one",
    )
)

assert not test_episode_analysis[
    CONCENTRATION_COLUMNS + ["episode_type"]
].isna().any().any()

## Episode types

In [37]:
# Mutually exclusive episode types
episode_type_performance = (
    test_episode_analysis
    .groupby("episode_type")
    .agg(
        n=("episode_id", "size"),
        independent_ndcg=("independent_ndcg10", "mean"),
        episode_context_ndcg=("context_ndcg10", "mean"),
        ndcg_difference=("ndcg10_difference", "mean"),
    )
    .reset_index()
)

episode_type_performance

,episode_type,n,independent_ndcg,episode_context_ndcg,ndcg_difference
0,Exact duplicates + additional product-code gro...,3228,0.424208,0.703858,0.279650
1,Exact duplicates only,3845,0.402094,0.535884,0.133790
2,No narrow grouping,8956,0.374126,0.486279,0.112153
3,Product-code grouping without exact duplicates,3971,0.431917,0.754987,0.323070


## Episode sizes

In [38]:
# Episode size
test_episode_analysis["episode_size_group"] = pd.cut(
    test_episode_analysis["episode_size"],
    bins=[1, 2, 4, 6, 10, np.inf],
    labels=["2", "3–4", "5–6", "7–10", "11+"],
)

episode_size_summary = (
    test_episode_analysis
    .groupby("episode_size_group", observed=True)
    .agg(
        n=("episode_id", "size"),
        independent_ndcg=("independent_ndcg10", "mean"),
        episode_context_ndcg=("context_ndcg10", "mean"),
        ndcg_difference=("ndcg10_difference", "mean"),
    )
)

episode_size_summary

,n,independent_ndcg,episode_context_ndcg,ndcg_difference
episode_size_group,,,,
2,2898,0.403505,0.558349,0.154844
3–4,5875,0.402277,0.587874,0.185597
5–6,3863,0.406013,0.593478,0.187465
7–10,4006,0.396015,0.594250,0.198236
11+,3358,0.385232,0.577924,0.192692


## Spearman correlations

In [39]:
# Spearman correlations with Benjamini-Hochberg FDR adjustment
correlation_rows = []

for feature in CONCENTRATION_COLUMNS:
    rho, p_value = spearmanr(
        test_episode_analysis[feature],
        test_episode_analysis["ndcg10_difference"],
    )
    correlation_rows.append({
        "feature": feature,
        "spearman_rho": rho,
        "raw_p": p_value,
    })

concentration_correlations = (
    pd.DataFrame(correlation_rows)
    .sort_values("spearman_rho", ascending=False)
    .reset_index(drop=True)
)

concentration_correlations["p_fdr"] = multipletests(
    concentration_correlations["raw_p"],
    method="fdr_bh",
)[1]

concentration_correlations

,feature,spearman_rho,raw_p,p_fdr
0,garment_group_concentration,0.430686,0.000000e+00,0.000000e+00
1,section_concentration,0.405479,0.000000e+00,0.000000e+00
2,department_concentration,0.401598,0.000000e+00,0.000000e+00
3,product_type_concentration,0.354014,0.000000e+00,0.000000e+00
4,product_group_concentration,0.343041,0.000000e+00,0.000000e+00
5,product_code_concentration,0.197877,1.009328e-175,1.513993e-175
6,graphical_appearance_concentration,0.098665,1.883679e-44,2.421873e-44
7,colour_concentration,0.065666,1.458475e-20,1.640785e-20
8,unique_article_concentration,0.034492,1.065795e-06,1.065795e-06


## OLS models

In [40]:
# Separate OLS models controlling for episode size, with customer-clustered SEs
controlled_rows = []

for feature in CONCENTRATION_COLUMNS:
    model = smf.ols(
        f"ndcg10_difference ~ {feature} + episode_size",
        data=test_episode_analysis,
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": test_episode_analysis["customer_id"]},
    )

    ci = model.conf_int().loc[feature]

    controlled_rows.append({
        "feature": feature,
        "coefficient": model.params[feature],
        "std_error": model.bse[feature],
        "p_value": model.pvalues[feature],
        "ci_lower": ci.iloc[0],
        "ci_upper": ci.iloc[1],
    })

controlled_concentration_results = (
    pd.DataFrame(controlled_rows)
    .sort_values("coefficient", ascending=False)
    .reset_index(drop=True)
)

controlled_concentration_results

,feature,coefficient,std_error,p_value,ci_lower,ci_upper
0,garment_group_concentration,0.517470,0.007290,0.000000e+00,0.503181,0.531759
1,department_concentration,0.498369,0.007632,0.000000e+00,0.483411,0.513328
2,section_concentration,0.498015,0.007500,0.000000e+00,0.483315,0.512715
3,product_group_concentration,0.432713,0.008166,0.000000e+00,0.416708,0.448717
4,product_type_concentration,0.423235,0.007780,0.000000e+00,0.407987,0.438483
5,product_code_concentration,0.342374,0.010750,1.366155e-222,0.321305,0.363444
6,graphical_appearance_concentration,0.143954,0.009408,7.496381e-53,0.125515,0.162393
7,unique_article_concentration,0.074189,0.015515,1.737351e-06,0.043780,0.104597
8,colour_concentration,0.066911,0.008837,3.689780e-14,0.049590,0.084232


In [41]:
# Combined post-hoc statistics
appendix_a7 = (
    concentration_correlations
    .merge(
        controlled_concentration_results,
        on="feature",
        how="outer",
        validate="one_to_one",
        suffixes=("_spearman", "_regression"),
    )
    .sort_values("spearman_rho", ascending=False)
    .reset_index(drop=True)
)

appendix_a7

,feature,spearman_rho,raw_p,p_fdr,coefficient,std_error,p_value,ci_lower,ci_upper
0,garment_group_concentration,0.430686,0.000000e+00,0.000000e+00,0.517470,0.007290,0.000000e+00,0.503181,0.531759
1,section_concentration,0.405479,0.000000e+00,0.000000e+00,0.498015,0.007500,0.000000e+00,0.483315,0.512715
2,department_concentration,0.401598,0.000000e+00,0.000000e+00,0.498369,0.007632,0.000000e+00,0.483411,0.513328
3,product_type_concentration,0.354014,0.000000e+00,0.000000e+00,0.423235,0.007780,0.000000e+00,0.407987,0.438483
4,product_group_concentration,0.343041,0.000000e+00,0.000000e+00,0.432713,0.008166,0.000000e+00,0.416708,0.448717
5,product_code_concentration,0.197877,1.009328e-175,1.513993e-175,0.342374,0.010750,1.366155e-222,0.321305,0.363444
6,graphical_appearance_concentration,0.098665,1.883679e-44,2.421873e-44,0.143954,0.009408,7.496381e-53,0.125515,0.162393
7,colour_concentration,0.065666,1.458475e-20,1.640785e-20,0.066911,0.008837,3.689780e-14,0.049590,0.084232
8,unique_article_concentration,0.034492,1.065795e-06,1.065795e-06,0.074189,0.015515,1.737351e-06,0.043780,0.104597


## LR Feature importance

In [42]:
# Logistic Regression feature importance
lr_context_importance = pd.DataFrame({
    "feature": EPISODE_CONTEXT_FEATURES,
    "standardized_coefficient": (
        context_model.named_steps["model"].coef_[0]
    ),
})

lr_context_importance["absolute_coefficient"] = (
    lr_context_importance["standardized_coefficient"].abs()
)

lr_context_importance = (
    lr_context_importance
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

lr_context_importance["rank"] = np.arange(
    1, len(lr_context_importance) + 1
)

lr_context_importance


,feature,standardized_coefficient,absolute_coefficient,rank
0,same_product_code_share,2.150810,2.150810,1
1,item_popularity,0.661932,0.661932,2
2,same_section_share,0.331169,0.331169,3
3,same_product_type_share,0.304984,0.304984,4
4,same_colour_share,0.229030,0.229030,5
5,same_department_share,0.180395,0.180395,6
6,customer_product_code_pref,0.165000,0.165000,7
7,price_distance,-0.160044,0.160044,8
8,same_appearance_share,0.157406,0.157406,9
9,same_garment_share,0.140194,0.140194,10


## LightGBM Robustness check

In [43]:
## LightGBM robustness check ##
train_lgbm = (
    train_features_400k
    .sort_values(["episode_id", "candidate_article"])
    .reset_index(drop=True)
)

validation_lgbm = (
    validation_features
    .sort_values(["episode_id", "candidate_article"])
    .reset_index(drop=True)
)

train_group = (
    train_lgbm.groupby("episode_id", sort=False).size().to_numpy()
)
validation_group = (
    validation_lgbm.groupby("episode_id", sort=False).size().to_numpy()
)

y_train_lgbm = train_lgbm["label"].astype("int32")
y_validation_lgbm = validation_lgbm["label"].astype("int32")

X_train_lgbm_independent = train_lgbm[
    INDEPENDENT_FEATURES
].astype("float32")
X_train_lgbm_context = train_lgbm[
    EPISODE_CONTEXT_FEATURES
].astype("float32")

X_val_lgbm_independent = validation_lgbm[
    INDEPENDENT_FEATURES
].astype("float32")
X_val_lgbm_context = validation_lgbm[
    EPISODE_CONTEXT_FEATURES
].astype("float32")

In [44]:
LGBM_PARAMS = {
    "objective": "lambdarank",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 50,
    "reg_lambda": 1.0,
    "lambdarank_truncation_level": 13,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

lgbm_independent = LGBMRanker(**LGBM_PARAMS)
lgbm_context = LGBMRanker(**LGBM_PARAMS)

common_fit_kwargs = {
    "group": train_group,
    "eval_group": [validation_group],
    "eval_names": ["validation"],
    "eval_metric": "ndcg",
    "eval_at": [10],
    "callbacks": [
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
        ),
        lgb.log_evaluation(50),
    ],
}

lgbm_independent.fit(
    X_train_lgbm_independent,
    y_train_lgbm,
    eval_set=[(X_val_lgbm_independent, y_validation_lgbm)],
    **common_fit_kwargs,
)

lgbm_context.fit(
    X_train_lgbm_context,
    y_train_lgbm,
    eval_set=[(X_val_lgbm_context, y_validation_lgbm)],
    **common_fit_kwargs,
)

c:\Users\stine\hm-bracketing\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	validation's ndcg@10: 0.364881
[100]	validation's ndcg@10: 0.370501
[150]	validation's ndcg@10: 0.375406
[200]	validation's ndcg@10: 0.378948
[250]	validation's ndcg@10: 0.38176
[300]	validation's ndcg@10: 0.383951
[350]	validation's ndcg@10: 0.385012
[400]	validation's ndcg@10: 0.386061
[450]	validation's ndcg@10: 0.388329
[500]	validation's ndcg@10: 0.389565
[550]	validation's ndcg@10: 0.390808
[600]	validation's ndcg@10: 0.393194
[650]	validation's ndcg@10: 0.394313
[700]	validation's ndcg@10: 0.395907
[750]	validation's ndcg@10: 0.396581
[800]	validation's ndcg@10: 0.398484
[850]	validation's ndcg@10: 0.400282
[900]	validation's ndcg@10: 0.401095
[950]	validation's ndcg@10: 0.402042
[1000]	validation's ndcg@10: 0.402999
Did not meet early stopping. Best iteration is:
[999]	validation's ndcg@10: 0.403018
Evaluated only: ndcg@10


c:\Users\stine\hm-bracketing\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	validation's ndcg@10: 0.584188
[100]	validation's ndcg@10: 0.590668
[150]	validation's ndcg@10: 0.594573
[200]	validation's ndcg@10: 0.597677
[250]	validation's ndcg@10: 0.600352
[300]	validation's ndcg@10: 0.601233
[350]	validation's ndcg@10: 0.601956
[400]	validation's ndcg@10: 0.602665
[450]	validation's ndcg@10: 0.603151
[500]	validation's ndcg@10: 0.604524
[550]	validation's ndcg@10: 0.605538
[600]	validation's ndcg@10: 0.60602
[650]	validation's ndcg@10: 0.606904
[700]	validation's ndcg@10: 0.607678
[750]	validation's ndcg@10: 0.60839
[800]	validation's ndcg@10: 0.608979
[850]	validation's ndcg@10: 0.609746
[900]	validation's ndcg@10: 0.609925
[950]	validation's ndcg@10: 0.610411
[1000]	validation's ndcg@10: 0.610927
Did not meet early stopping. Best iteration is:
[1000]	validation's ndcg@10: 0.610927
Evaluated only: ndcg@10


,learning_rate,0.05
,n_estimators,1000
,objective,'lambdarank'
,min_child_samples,50
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,lambdarank_truncation_level,13
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31


## LightGBM fresh test

In [ ]:
## Fresh LightGBM test holdout (38,085 eligible test episodes not used for the LR final test)
used_lr_test_ids = set(test_prepared_sample["episode_id"])

lgbm_test_prepared = test_prepared[
    ~test_prepared["episode_id"].isin(used_lr_test_ids)
].copy()

print(f"Fresh LightGBM test episodes: {len(lgbm_test_prepared):,}")  # 38,085

lgbm_test_candidates = build_candidate_df(
    lgbm_test_prepared,
    candidate_catalog,
    n_negatives=99,
    seed=RANDOM_SEED,
)

lgbm_test_features = add_context_features(
    add_independent_features(lgbm_test_candidates),
    lgbm_test_prepared,
)

lgbm_test = (
    lgbm_test_features
    .sort_values(["episode_id", "candidate_article"])
    .reset_index(drop=True)
)

lgbm_test_group = (
    lgbm_test.groupby("episode_id", sort=False).size().to_numpy()
)
assert np.all(lgbm_test_group == 100)

Fresh LightGBM test episodes: 38,085


In [46]:
lgbm_scores = lgbm_test[
    ["episode_id", "customer_id", "candidate_article", "label"]
].copy()

lgbm_scores["independent_score"] = lgbm_independent.predict(
    lgbm_test[INDEPENDENT_FEATURES].astype("float32"),
    num_iteration=lgbm_independent.best_iteration_,
)

lgbm_scores["context_score"] = lgbm_context.predict(
    lgbm_test[EPISODE_CONTEXT_FEATURES].astype("float32"),
    num_iteration=lgbm_context.best_iteration_,
)

for model in ["independent", "context"]:
    lgbm_scores[f"{model}_rank"] = (
        lgbm_scores.groupby("episode_id")[f"{model}_score"]
        .rank(ascending=False, method="first")
    )

lgbm_targets = lgbm_scores.loc[lgbm_scores["label"] == 1].copy()

for model in ["independent", "context"]:
    hit, mrr, ndcg = calculate_ranking_metrics(
        lgbm_targets[f"{model}_rank"], k=10
    )
    lgbm_targets[f"{model}_hit10"] = hit
    lgbm_targets[f"{model}_mrr10"] = mrr
    lgbm_targets[f"{model}_ndcg10"] = ndcg

lgbm_results = pd.DataFrame({
    "Independent": [
        lgbm_targets["independent_hit10"].mean(),
        lgbm_targets["independent_mrr10"].mean(),
        lgbm_targets["independent_ndcg10"].mean(),
    ],
    "Episode-context": [
        lgbm_targets["context_hit10"].mean(),
        lgbm_targets["context_mrr10"].mean(),
        lgbm_targets["context_ndcg10"].mean(),
    ],
}, index=["Hit@10", "MRR@10", "NDCG@10"])

lgbm_results["Absolute difference"] = (
    lgbm_results["Episode-context"] - lgbm_results["Independent"]
)

lgbm_targets, lgbm_cluster_ci = clustered_ci_table(lgbm_targets)

lgbm_results

,Independent,Episode-context,Absolute difference
Hit@10,0.682132,0.817487,0.135355
MRR@10,0.401870,0.588023,0.186154
NDCG@10,0.468265,0.642950,0.174685


In [47]:
lgbm_cluster_ci

,Metric,Difference,CI lower,CI upper
0,Hit@10,0.135355,0.131005,0.139888
1,MRR@10,0.186154,0.182028,0.190466
2,NDCG@10,0.174685,0.171029,0.178652


## LightGBM feature importance

In [48]:
## LightGBM feature importance ##
lgbm_context_importance = pd.DataFrame({
    "feature": EPISODE_CONTEXT_FEATURES,
    "gain_importance": lgbm_context.booster_.feature_importance(
        importance_type="gain"
    ),
    "split_importance": lgbm_context.booster_.feature_importance(
        importance_type="split"
    ),
})

lgbm_context_importance["gain_percent"] = (
    lgbm_context_importance["gain_importance"]
    / lgbm_context_importance["gain_importance"].sum()
    * 100
)

lgbm_context_importance = (
    lgbm_context_importance
    .sort_values("gain_importance", ascending=False)
    .reset_index(drop=True)
)

context_gain_share = (
    lgbm_context_importance.loc[
        lgbm_context_importance["feature"].isin(CONTEXT_FEATURES),
        "gain_importance",
    ].sum()
    / lgbm_context_importance["gain_importance"].sum()
)

print(f"Context features: {context_gain_share:.1%} of total LightGBM gain")
lgbm_context_importance

Context features: 58.5% of total LightGBM gain


,feature,gain_importance,split_importance,gain_percent
0,item_popularity,4.146570e+06,7068,31.367126
1,same_section_share,2.966870e+06,981,22.443169
2,same_department_share,2.834159e+06,472,21.439266
3,same_product_type_share,7.779792e+05,1072,5.885098
4,customer_section_pref,6.082565e+05,1575,4.601214
5,same_product_code_share,4.039508e+05,241,3.055724
6,same_colour_share,3.209945e+05,976,2.428193
7,same_garment_share,2.470365e+05,729,1.868731
8,customer_appearance_pref,1.652724e+05,1884,1.250219
9,same_appearance_share,1.459172e+05,1086,1.103805


## LensKit verification

In [49]:
## LensKit verification ##
from lenskit.data import ItemListCollection
from lenskit.metrics import MeasurementCollector
from lenskit.metrics.ranking import Hit, NDCG, RecipRank, LogRankWeight


def evaluate_with_lenskit(candidate_df, rank_col, k=10):
    required = {
        "episode_id",
        "candidate_article",
        "label",
        rank_col,
    }
    missing = required - set(candidate_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    assert (
        candidate_df.groupby("episode_id")["label"]
        .sum()
        .eq(1)
        .all()
    )

    rec_df = (
        candidate_df[
            ["episode_id", "candidate_article", rank_col]
        ]
        .rename(columns={
            "candidate_article": "item_id",
            rank_col: "rank",
        })
        .sort_values(["episode_id", "rank"], kind="stable")
        .reset_index(drop=True)
    )

    truth_df = (
        candidate_df.loc[
            candidate_df["label"] == 1,
            ["episode_id", "candidate_article"],
        ]
        .rename(columns={"candidate_article": "item_id"})
        .reset_index(drop=True)
    )

    recs = ItemListCollection.from_df(rec_df, key="episode_id")
    truth = ItemListCollection.from_df(truth_df, key="episode_id")

    mc = MeasurementCollector()
    mc.add_metric(Hit(n=k))
    mc.add_metric(RecipRank(n=k))
    mc.add_metric(
        NDCG(
            n=k,
            weight=LogRankWeight(base=2, offset=1),
        )
    )

    summary, per_episode = mc.measure_run(recs, truth)

    hit_col = next(c for c in per_episode.columns if c.startswith("Hit"))
    rr_col = next(c for c in per_episode.columns if c.startswith("RecipRank"))
    ndcg_col = next(c for c in per_episode.columns if c.startswith("NDCG"))

    results = pd.Series({
        f"Hit@{k}": per_episode[hit_col].mean(),
        f"MRR@{k}": per_episode[rr_col].mean(),
        f"NDCG@{k}": per_episode[ndcg_col].mean(),
    })

    return results

In [50]:
lr_lenskit_ind = evaluate_with_lenskit(
    test_scores,
    rank_col="independent_rank",
)
lr_lenskit_context = evaluate_with_lenskit(
    test_scores,
    rank_col="context_rank",
)

lgbm_lenskit_ind = evaluate_with_lenskit(
    lgbm_scores,
    rank_col="independent_rank",
)
lgbm_lenskit_context = evaluate_with_lenskit(
    lgbm_scores,
    rank_col="context_rank",
)

lenskit_verification = pd.DataFrame([
    {
        "Algorithm": "Logistic Regression",
        "Model": "Independent-choice",
        "Metric": metric,
        "Original": test_results.loc[metric, "Independent"],
        "LensKit": lr_lenskit_ind[metric],
    }
    for metric in ["Hit@10", "MRR@10", "NDCG@10"]
] + [
    {
        "Algorithm": "Logistic Regression",
        "Model": "Episode-context",
        "Metric": metric,
        "Original": test_results.loc[metric, "Episode-context"],
        "LensKit": lr_lenskit_context[metric],
    }
    for metric in ["Hit@10", "MRR@10", "NDCG@10"]
] + [
    {
        "Algorithm": "LightGBM",
        "Model": "Independent-choice",
        "Metric": metric,
        "Original": lgbm_results.loc[metric, "Independent"],
        "LensKit": lgbm_lenskit_ind[metric],
    }
    for metric in ["Hit@10", "MRR@10", "NDCG@10"]
] + [
    {
        "Algorithm": "LightGBM",
        "Model": "Episode-context",
        "Metric": metric,
        "Original": lgbm_results.loc[metric, "Episode-context"],
        "LensKit": lgbm_lenskit_context[metric],
    }
    for metric in ["Hit@10", "MRR@10", "NDCG@10"]
])

lenskit_verification["Discrepancy"] = (
    lenskit_verification["LensKit"] - lenskit_verification["Original"]
)

assert np.allclose(
    lenskit_verification["Original"],
    lenskit_verification["LensKit"],
    atol=1e-6,
)

lenskit_verification

,Algorithm,Model,Metric,Original,LensKit,Discrepancy
0,Logistic Regression,Independent-choice,Hit@10,0.605600,0.605600,0.0
1,Logistic Regression,Independent-choice,MRR@10,0.335476,0.335476,0.0
2,Logistic Regression,Independent-choice,NDCG@10,0.399060,0.399060,0.0
3,Logistic Regression,Episode-context,Hit@10,0.765200,0.765200,0.0
4,Logistic Regression,Episode-context,MRR@10,0.527718,0.527718,0.0
5,Logistic Regression,Episode-context,NDCG@10,0.584285,0.584285,0.0
6,LightGBM,Independent-choice,Hit@10,0.682132,0.682132,0.0
7,LightGBM,Independent-choice,MRR@10,0.401870,0.401870,0.0
8,LightGBM,Independent-choice,NDCG@10,0.468265,0.468265,0.0
9,LightGBM,Episode-context,Hit@10,0.817487,0.817487,0.0


## Exporting result tables

In [51]:
## Exporting all result tables ##
# Table 1 — chronological split
table1_split = pd.DataFrame([
    {
        "Split": name,
        "Percentage": len(df) / len(experiment_episodes) * 100,
        "From": df["t_dat"].min().date(),
        "Until": df["t_dat"].max().date(),
        "Shopping episodes": len(df),
    }
    for name, df in splits.items()
])
table1_split.to_csv(
    RESULTS_DIR / "table_01_chronological_split.csv",
    index=False,
)

# Validation results used during model development
validation_table = (
    validation_results
    .rename_axis("Metric")
    .reset_index()
    .merge(validation_cluster_ci, on="Metric", how="left")
)
validation_table.to_csv(
    RESULTS_DIR / "validation_random_negative_results.csv",
    index=False,
)

# Table 7 — final Logistic Regression test
table7_lr_final = (
    test_results
    .rename_axis("Metric")
    .reset_index()
    .merge(test_cluster_ci, on="Metric", how="left")
)
table7_lr_final.to_csv(
    RESULTS_DIR / "table_07_lr_final_test.csv",
    index=False,
)

# Table 8 — hard-negative robustness
hard_negative_comparison.to_csv(
    RESULTS_DIR / "table_08_hard_negative_comparison.csv",
    index=False,
)

hard_results_full = (
    hard_results
    .rename_axis("Metric")
    .reset_index()
    .merge(hard_cluster_ci, on="Metric", how="left")
)
hard_results_full.to_csv(
    RESULTS_DIR / "hard_negative_detailed_results.csv",
    index=False,
)

# Table 9 — LightGBM final test
table9_lgbm_final = (
    lgbm_results
    .rename_axis("Metric")
    .reset_index()
    .merge(lgbm_cluster_ci, on="Metric", how="left")
)
table9_lgbm_final.to_csv(
    RESULTS_DIR / "table_09_lgbm_final_test.csv",
    index=False,
)

# Tables 10–13
episode_type_performance.to_csv(
    RESULTS_DIR / "table_10_episode_type_performance.csv",
    index=False,
)

episode_size_summary.reset_index().to_csv(
    RESULTS_DIR / "table_11_episode_size_performance.csv",
    index=False,
)

controlled_concentration_results.to_csv(
    RESULTS_DIR / "table_12_controlled_concentration_regressions.csv",
    index=False,
)

concentration_correlations.to_csv(
    RESULTS_DIR / "table_13_spearman_concentration.csv",
    index=False,
)

# Appendix A1 — Logistic Regression feature importance
lr_context_importance.to_csv(
    RESULTS_DIR / "appendix_A1_lr_feature_importance.csv",
    index=False,
)

# Appendix A2 — LightGBM feature importance
lgbm_context_importance.to_csv(
    RESULTS_DIR / "appendix_A2_lgbm_feature_importance.csv",
    index=False,
)

# Appendix A5 — experimental sample flow
appendix_a5 = pd.DataFrame({
    "Processing stage": [
        "Chronological split",
        "After warm-up-customer eligibility",
        "After eligible-target preparation",
        "Final LR experimental sample",
        "Fresh LightGBM test holdout",
    ],
    "Training": [
        len(train_episodes),
        len(train_episodes_warm),
        len(train_prepared),
        len(train_prepared_400k),
        np.nan,
    ],
    "Validation": [
        len(validation_episodes),
        len(validation_episodes_warm),
        len(validation_prepared),
        len(validation_prepared_sample),
        np.nan,
    ],
    "Test": [
        len(test_episodes),
        len(test_episodes_warm),
        len(test_prepared),
        len(test_prepared_sample),
        len(lgbm_test_prepared),
    ],
})
appendix_a5.to_csv(
    RESULTS_DIR / "appendix_A5_experimental_sample_flow.csv",
    index=False,
)

# Appendix A6 — LensKit verification
lenskit_verification.to_csv(
    RESULTS_DIR / "appendix_A6_lenskit_verification.csv",
    index=False,
)

# Appendix A7 — full post-hoc statistics
appendix_a7.to_csv(
    RESULTS_DIR / "appendix_A7_posthoc_statistics.csv",
    index=False,
)

# Table 14 — union of top five features from either model
lr_for_merge = lr_context_importance[
    ["feature", "standardized_coefficient", "rank"]
].rename(columns={"rank": "LR rank"})

lgbm_for_merge = lgbm_context_importance[
    ["feature", "gain_percent"]
].copy()
lgbm_for_merge["LGBM rank"] = np.arange(
    1, len(lgbm_for_merge) + 1
)

table14_all = lr_for_merge.merge(
    lgbm_for_merge,
    on="feature",
    how="inner",
    validate="one_to_one",
)

table14_all["Feature type"] = np.where(
    table14_all["feature"].isin(CONTEXT_FEATURES),
    "Episode context",
    "Historical",
)

top_union = (
    set(table14_all.nsmallest(5, "LR rank")["feature"])
    | set(table14_all.nsmallest(5, "LGBM rank")["feature"])
)

table14 = (
    table14_all.loc[table14_all["feature"].isin(top_union)]
    .sort_values("LGBM rank")
    .reset_index(drop=True)
)

table14.to_csv(
    RESULTS_DIR / "table_14_feature_importance_comparison.csv",
    index=False,
)

print("\nAll report result tables written to:", RESULTS_DIR)
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print("-", path.name)



All report result tables written to: C:\Users\stine\hm-bracketing\results
- appendix_A1_lr_feature_importance.csv
- appendix_A2_lgbm_feature_importance.csv
- appendix_A5_experimental_sample_flow.csv
- appendix_A6_lenskit_verification.csv
- appendix_A7_posthoc_statistics.csv
- hard_negative_detailed_results.csv
- lr_final_test_clustered_ci.csv
- lr_final_test_results.csv
- table_01_chronological_split.csv
- table_03_episode_characteristics.csv
- table_04_within_episode_concentration.csv
- table_05_episode_type_frequencies.csv
- table_06_strong_grouping.csv
- table_07_lr_final_test.csv
- table_08_hard_negative_comparison.csv
- table_09_lgbm_final_test.csv
- table_10_episode_type_performance.csv
- table_11_episode_size_performance.csv
- table_12_controlled_concentration_regressions.csv
- table_13_spearman_concentration.csv
- table_14_feature_importance_comparison.csv
- validation_random_negative_results.csv
